# DRISHTI — RELLIS-3D download: Ouster OS1-64 stream

Ticket #3 (⬜ DETACHABLE — runs for hours, blocks nothing else; kick it off and keep working on other tickets).

This is the **primary** stream — 64-channel Ouster OS1, matching the sensor config `configs/sensor_ouster_os1_64.yaml`. It pulls the official RELLIS-3D SemanticKITTI-format archive from Google Drive and keeps only sequence `00004`.

**Standalone notebook** — does not require `colab_setup.ipynb` to have run first, but does need the repo, which this notebook clones if missing.

**Size/time reality check** (Build Map Ticket #3): the KITTI-format archive is **not** published per-sequence — it's one combined 14GB file across all 5 sequences, plus 174MB labels and 174MB poses (~14.35GB transient download). You keep only the `00004/` slice afterward, but Colab/Drive has to hold the whole archive transiently to unzip it. Budget Drive space and time accordingly, and expect Colab to disconnect mid-transfer at least once on a download this size — just re-run the download cell if it does.

If you'd rather start with the lighter 32-channel Velodyne stream (5.58GB) instead, use `colab_download_rellis_vel.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone once per session; re-running is safe (pulls instead of re-cloning).
import os
REPO_URL = 'REPLACE_WITH_YOUR_GITHUB_URL'  # push this repo to GitHub first, per Ticket #1 "Watch out"
if not os.path.isdir('drishti'):
    !git clone {REPO_URL} drishti
%cd drishti
!git pull

In [ ]:
!pip install -q gdown

## The download

Downloads the 14GB Ouster SemanticKITTI archive + 174MB labels + 174MB poses to `/content/drive/MyDrive/drishti_data/rellis_raw/`, extracts only `00004/`, and copies the result to `/content/drive/MyDrive/drishti_data/rellis/00004/` — where `perception/rellis_loader.py` expects it.

In [ ]:
DEST_ROOT = '/content/drive/MyDrive/drishti_data'
!bash scripts/download_rellis.sh {DEST_ROOT} os1

## Verify

Confirm the extraction landed where the loader expects it, then run the real-sequence test (Build Map Ticket #3's own "Done when" condition).

In [ ]:
import os
seq_dir = f'{DEST_ROOT}/rellis/00004'
for sub in ('os1_cloud_node_kitti_bin', 'os1_cloud_node_semantickitti_label_id', 'poses.txt'):
    p = os.path.join(seq_dir, sub)
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')
!ls -la {seq_dir} 2>/dev/null | head -20

In [ ]:
import os
os.environ['RELLIS_SEQ_DIR'] = f'{DEST_ROOT}/rellis/00004'  # tests/test_rellis_loader.py reads this exact var
!pytest -q tests/test_rellis_loader.py::test_real_sequence_00004 -v

## Cleanup (optional, once the above passes)

`rellis_raw/` holds the full transient archives (14GB+) — you only need `rellis/00004/` going forward. Delete `rellis_raw/` to free Drive space; do **not** delete `rellis/00004/`.

In [ ]:
# Uncomment once you've verified the cells above passed:
# import shutil
# shutil.rmtree(f'{DEST_ROOT}/rellis_raw', ignore_errors=True)